# T02: Regular Languages

## Language Processing

## LEI/2025-26

#### Nuno Macedo
Universidade do Minho

# 2.1 Regular Languages

- Regular languages is the simplest class of languages in Chomsky hierarchy

- They cannot express dependencies like "the same number of symbols" or properly nested structures

- Recognizable by **finite automata**

- Described formally by **regular expressions** (REs)

- Important for **lexical analysis**, input validation, and simple pattern recognition


## Example: A Language of Valid Identifiers

- The language of Python identifiers
  - ✅ `aux_function`, `_hiddenVar`, `xy23`
  - ❌ `23xy`, `ab+xy`, `-xyz`



In [ ]:
aux_function = True
_hiddenVar = True
xy23 = True
# 23xy = True
# ab+xy = True
# -xyz = True

- Pattern description:
  - *Arbitrary-length* sequences of *alphanumeric or underscore* symbols
  - *Cannot start* with a numeric symbol (why?)

- How can we **formally specify** these rules?

- How can we **efficiently recognize** such identifiers?

# 2.2 Regular Expressions
   
- **Regular expressions** (REs), also known as *regexes*, are human-friendly ways to describe regular languages

- Main uses:
  - Define lexing rules for compilers/interpreters
  - Detect patterns in text processing
  - Search and replace in editors or IDEs



## Regular expressions in theory

* A RE is built from:

  - **Base cases**:
    - Symbols $a \in \Sigma$
    - Empty set: $\emptyset$
    - Empty word: $\epsilon$

  - **Inductive cases** (for expressions $e_1$, $e_2$):
    - Concatenation: $e_1 e_2$
    - Union / alternation: $e_1 | e_2$
    - Kleene closure: $e_1^*$ (zero or more repetitions)

## Example: A Language of Valid Identifiers

- $(a|b|...|w|z|A|B|...|W|Z|\_)(a|b|...|w|z|A|B|...|W|Z|\_|0|1|...|9)^*$

  - $(a|b|...|w|z|A|B|...|W|Z|\_)$ any alphabetic symbol or an underscore

  - $(a|b|...|w|z|A|B|...|W|Z|\_|0|1|...|9)$ any alpha-numeric symbol or an underscore

  - $(a|b|...|w|z|A|B|...|W|Z|\_|0|1|...|9)^*$ any alpha-numeric symbol or an underscore, repeated an arbitrary number of times

  - $(a|b|...|w|z|A|B|...|W|Z|\_)(a|b|...|w|z|A|B|...|W|Z|\_|0|1|...|9)^*$ any alpha-numeric symbol or an underscore, followed by an arbitrary number of alpha-numeric symbols or underscores



## Concrete syntax for regular expressions

- Theory of computation deals with the formal notion of REs

- RE processors in introduce other operators to facilitate the writing of REs
  - In most cases, syntactic sugar that does not increase theoretical expressivity

- Syntax may vary slightly depending on the RE processor
  - A popular syntax is the one inspired by Perl, which is followed by Python

- Special symbols
  - `.` matches any symbol except a new line
  - `^` matches the start of the string
  - `$` matches the end of the string

- Symbol repetitions
  - `*` matches any repetition of the preceeding regex
  - `+` matches one or more repetitions of the preceeding regex
  - `?` matches 0 or 1 repetitions of the preceeding regex
  - `{n}` matches exactly `n` repetitions of the preceeding regex
  - `{m,n}` matches between `n` and `m` repetitions of the preceeding regex

- `|` denotes the union (alternative) of regexes

- `[...]` creates a group of alternative symbols
  - `-` can be used for ranges of symbols (to match `-`, put it in the begining or end of group)
  - `^` denotes the complement, all symbols except those declared
  - All symbols treated as literal inside group except `-`, `]` and `^`
  
- Symbol groups
  - `\d` for any digit symbol
  - `\s` for any whitespace symbol
  - `\w` any alpha-numberic symbol plus the underscore `_`
  - `\b` matches the boundary of words without consuming any symbol
  - Upper-case for complement set (e.g., `\D` is any symbol that is not a digit)

- From now on, we will use this syntax, not the formal RE notation

## Example: A Language of Valid Identifiers (revisited)

- `[a-zA-Z_][a-zA-Z0-9_]*`

  - `a-z` all symbols between `a` and `z`

  - `A-Z` all symbols between `A` and `Z`

  - `0-9` all symbols between `0` and `9`

  - `[a-zA-Z_]` any symbol between `a` and `z`, `A` and `Z` or underscore `_`

  - `[a-zA-Z0-9_]` any symbol between `a` and `z`, `A` and `Z`, `0` and `9` or underscore `_`

  - `[a-zA-Z0-9_]*` any repetitions of symbols between `a` and `z`, `A` and `Z`, `0` and `9` or underscore `_`

- Or simply `[a-zA-Z_]\w*`


## Example: A Language for Floating Point Numbers

In [ ]:
print(10.34)
print(.34)
print(10.)
# print(.)
print(-.34)
print(+10.)
print(10.34E-3)
print(.34e3)

10.34
0.34
10.0
-0.34
10.0
0.01034
340.0


- Unsigned integer numbers:

    `\d+` → `23`

- Unsigned floating point numbers:

    `\d+\.\d+`  → `23.45`

- Unsigned integer or floating point numbers:

    `\d+(\.\d+)?` → `23` or `23.45`

- Unsigned integer or floating point numbers, possibly omitting one side of point but not both:

    `\d+(\.\d*)?|\.\d+` → `23` or `23.45` or `.45` or `23.`

- Possibly signed integer or floating point numbers, possibly omitting one side of point but not both:

     `[+-]?(\d+(\.\d*)?|\.\d+)` → `+23` or `-23.45` or `.45` or `+23.`

- Possibly signed floating point numbers, possibly omitting one side of point but not both, with optional, possibly signed, exponent:
    
    `[+-]?(\d+\.\d*|\d*\.\d+)([eE][+-]?\d+)?` → `+23` or `-23.45E+56` or `.45e-56` or `+23.e56`

## Beyond word recognition

- Although REs in theory are used to recognize words belonging to a language, in practice they are often used find substrings in a larger text

- In this case, the output is no longer yes/no, but a set of **matches**

- The language accepted by the RE does not change, but the set of returned matches may change depending on the RE processor

- This is the basis of the tokenization stage in the language processing pipeline

## Greedy vs. Lazy matching

- Theoretical REs are only concerned about recognition
  - Whether a word belongs to the language or not

- But the same word may be matched by a RE in different ways
  - This is not irrelevant for the user when processing a text

- By default, repetition operators are *greedy*, the match as many symbols as possible

- *Lazy* variants are used when a `?` is appended (`*?`, `+?`, `{n,m}?`)

- This affects how the RE processor matches, but not the overall language

- *Example*: detecting text between quotes in input text `I say "Hello" and you say "World"`
  - Regex `".*"` is greedy, it has a single match, `"Hello" and you say "World"`
  -  Regex `".*?"` is lazy, it has two matches, both `"Hello"` and `"World"`

## Capturing groups

- Often we are only interested in part of the matched substring

- REs support the introduction of *capturing groups* using parenthesis `(...)`, that identify what is effectively returned in the match

- *Example*: detecting text between quotes in input text `I say "Hello" and you say "World"`
  - finding text between quotes with `".*?"`, match will include the quotes, `"Hello"` and `"World"`
  - if we are only interested in the text within the quotes, we create a capturing group `"(.*?)"`, will only match `Hello` and `World`

- Sometimes we need to use parenthesis to group expressions but do not want a capturing group
  - In that case we have to escape the delimiters with `(?:...)`

## Backreferences

- Modern REs also allow *backreferences* to captured groups with `\number`

- This is a powerful feature that goes beyond what theoretical REs allow

- *Example*: find all repeated alpha-numeric patterns
  - `(\w+)\1` matches words like `AA`, `ABAB` or `ABCABC`

- Also useful for text substitution that preserves part of the original match

## Exercise: Find title patterns in JSON file

Open file `cinema.json` in any text editor of your choice.

- Find all occurrences of:
  - All titles
  - All titles that include a number
  - All titles that include a **real** number
  - All titles that include **two** numbers
  - All titles where character `x` **does not** occur
  - All titles with **repeated** words
  - All **multi-line** titles

- Remove all subtitles (identified by `:`)

- Surround numbers in titles by single quotes `'`

## Up next week:
  
  - Regular expressions in Python, module `re`
  - Finite automata for the recognition of REs
  - Translations between REs and automata

-- Nuno Macedo